In [1]:
# 1: read the entire book as plain text

import numpy as np
with open("mysterious_island.txt", encoding = "utf8") as fp:
    text = fp.read()

# Remove portions from beginning and end.
start_indx = text.find("THE MYSTERIOUS ISLAND")
end_indx = text.find("End of the project Gutenberg")
text = text[start_indx:end_indx]
char_set = set(text)

print("Total Length: " + str(len(text)))
print("Unique Characters: " + str(len(char_set)))


Total Length: 1131145
Unique Characters: 84


In [3]:
# 2: Convert text to numeric format


# Sort unique characters alphabetical order
chars_sorted = sorted(char_set)
# Create dictionary that maps each char to integer
char2int = {ch:i for i, ch in enumerate(chars_sorted)}

# Reverse mapping with array
char_array = np.array(chars_sorted)

# encode the text using the dictionary
text_encoded = np.array([char2int[ch] for ch in text])

# THE MYSTERIOUS
print(text[:15], '     == Encoding ==> ', text_encoded[:15])
# ISLAND
print(text_encoded[15:21], ' == Reverse  ==> ', ''.join(char_array[text_encoded[15:21]]))

THE MYSTERIOUS       == Encoding ==>  [47 35 32  1 40 52 46 47 32 45 36 42 48 46  1]
[36 46 39 28 41 31]  == Reverse  ==>  ISLAND


In [5]:
# print mappings of first five characters
for ex in text_encoded[:5]:
    print("{} -> {}".format(ex, char_array[ex]))

47 -> T
35 -> H
32 -> E
1 ->  
40 -> M


In [7]:
# 3: Create text chunks 41 chars each
import torch
from torch.utils.data import Dataset

# Declare sequence length as 40
seq_length = 40
# Declare chunk size as seq_length + 1
chunk_size = seq_length + 1
# Declare text_chunks array
text_chunks = [text_encoded[i:i+chunk_size] for i in range(len(text_encoded)-chunk_size + 1)]

# Declare TextDataset class
# To convert text_chunks array into object
class TextDataset(Dataset):
    # CONSTRUCTOR
    def __init__(self, text_chunks):
        # Text chunks attribute
        self.text_chunks = text_chunks
    def __len__(self):
        # return length of text_chunks
        return len(self.text_chunks)
    def __getitem__(self, idx):
        text_chunk = self.text_chunks[idx]
        # return sequence and target
        return text_chunk[:,-1].long(), text_chunk[1:].long()


seq_dataset = TextDataset(torch.tensor(text_chunks))

# Looking at example sequences:
for i, (seq, target) in enumerate(seq_dataset):
    print(' Input (x):', repr(''.join(char_array[seq])))
    print('Target (y):', repr(''.join(char_array[target])))
    print()
    if i == 1:
        break

C:\Users\tobia\AppData\Local\Temp\ipykernel_27220\1623620977.py:28: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\cb\pytorch_1000000000000\work\torch\csrc\utils\tensor_new.cpp:281.)
  seq_dataset = TextDataset(torch.tensor(text_chunks))


In [9]:
# 4: shuffle and batch with DataLoader

from torch.utils.data import DataLoader
batch_size = 64
torch.manual_seed(1)
seq_dl = DataLoader(seq_dataset, batch_size = batch_size, shuffle = True, drop_last = True)

In [11]:
# 5: Create RNN model

import torch.nn as nn
class RNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, rnn_hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.rnn_hidden_size = rnn_hidden_size
        self.rnn = nn.LSTM(embed_dim, rnn_hidden_size, batch_first = True)
        self.fc = nn.Linear(rnn_hidden_size, vocab_size)

    def forward(self, x, hidden, cell):
        out = self.embedding(x).unsqueeze(1)
        out, (hidden,cell) = self.rnn(out, (hidden, cell))
        out = self.fc(out).reshape(out.size(0), -1)
        return out, hidden, cell
    # Method to initialize the parameters for the LSTM
    def init_hidden(self, batch_size):
        hidden = torch.zeros(1, batch_size, self.rnn_hidden_size)
        cell = torch.zeros(1, batch_size, self.rnn_hidden_size)
        return hidden, cell

# specify model parameters
vocab_size = len(char_array)
embed_dim = 256
rnn_hidden_size = 512
torch.manual_seed(1)
model = RNN(vocab_size, embed_dim, rnn_hidden_size)


In [13]:
# 6: Loss function and optimizer
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr = 0.005)

In [17]:
# 7: Train for 10000 epochs
num_epochs = 10000
torch.manual_seed(1)
# For each epoch
for epoch in range(num_epochs):
    # initialize LSTM parameters
    hidden, cell = model.init_hidden(batch_size)
    # Get the current batch of data
    seq_batch, target_batch = next(iter(seq_dl))
    optimizer.zero_grad()
    # Initialize loss parameter
    loss = 0
    # for each 
    for c in range(seq_length):
        if c <= 10:
            print(seq_batch[:, c])
        pred, hidden, cell = model(seq_batch[:, c], hidden, cell)
        loss += loss_fn(pred, target_batch[:, c])
    loss.backward()
    optimizer.step()
    loss = loss.item()/seq_length
    if epoch % 500 == 0:
        print(f"Epoch {epoch} loss: {loss:.4f}")

IndexError: too many indices for tensor of dimension 1

In [ ]:
# Evaluation phase: New pass